## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 4,
        'suites': [{'cases': [{'code': ">>> t1a\n'E'", 'hidden': False, 'locked': False},
                              {'code': ">>> t1b\n'BUNNY'", 'hidden': False, 'locked': False},
                              {'code': ">>> t1c\n'BUNNY'", 'hidden': False, 'locked': False},
                              {'code': ">>> t1d\n'BUNNY'", 'hidden': False, 'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> t2a\n'A'", 'hidden': False, 'locked': False},
                              {'code': ">>> t2b\n'N'", 'hidden': False, 'locked': False},
                              {'code': ">>> t2c\n'B'", 'hidden': False, 'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> shift('Z', 1)\n'A'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> shift('Y', 2)\n'A'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> shift('A', 26)\n'A'",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> encode('A', 0)\n'A'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> encode('AA', 0)\n'AA'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> encode('ZABCDEFGHIJKLMNOPQRSTUVWXY', 1)\n"
                                       "'ABCDEFGHIJKLMNOPQRSTUVWXYZ'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> encode('A', 27)\n'B'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> encode('BUNNY', 17)\n'SLEEP'",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('09-Sequences', _TESTS)
check = grader.check

print("Ready. 09-Sequences loaded: 4 question(s), 10 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

# Homework: Sequences — Caesar Cipher

A Caesar Cipher is a simple substitution cipher encryption scheme where each letter in the original message is replaced by a letter a fixed number of positions away in the alphabet.

In our version only alphabet letters will be shifted, all other letters will be left un-encrypted and letters will wrap around from the end of the alphabet back to the front, i.e., the letter after "Z" will be "A". Further we will assume all messages are in capital letters. We will ignore (leave as is) spaces, punctuation and other characters not in [A-Z].

Suppose our shift is 3 and the message is "HELLO" then our encrypted message becomes "KHOOR":

- 'H' shifted by 3 becomes 'K'
- 'E' shifted by 3 becomes 'H'
- 'L' shifted by 3 becomes 'O'
- 'L' shifted by 3 becomes 'O'
- 'O' shifted by 3 becomes 'R'

We can decrypt a message by shifting the letters the other direction. In this case subtracting 3 from their position in the alphabet, noting that "Z" comes before "A" to handle wrapping.

Decrypting "KHOOR" with a shift of 3 we get "HELLO"

- 'K' shifted back by 3 becomes 'H'
- 'H' shifted back by 3 becomes 'E'
- 'O' shifted back by 3 becomes 'L'
- 'O' shifted back by 3 becomes 'L'
- 'R' shifted back by 3 becomes 'O'

# Task 1: Understanding the substitution cipher. 

A. If we encoded the message "A" by shifting each letter in the message by 4, then what is the encoded message?
    - Add your answer to the next cell in the variable `t1a = ""`

B. If we encoded the message "SLEEP" by shifting each letter in the message by 9, then what is the encoded message?
    - Add your answer to the variable `t1b = ` 

Suppose we already had methods to encrypt and decrypt messages: 

```python

def encrypt(msg, shift):
    """ encrypt msg by shifting the letters by shift positions

        msg: the string message to encrypt
        shift: the integer shift to shift each letter in msg by

        return: the encrypted string
    """

def decrypt(msg, shift):
    """ decrypt msg by shifting the letters backwards by shift positions

        msg: the encrypted string message to decrypt
        shift: the integer shift to shift each letter backwards in msg by

        return: the decrypted string
    """
    
```

C. What is the result of the following code: 
```python 
decrypt(encrypt("BUNNY", 17), 17)
```
    - Add your answer to the variable `t1c = `


D. What is the result of the following code:
```python 
encrypt(encrypt("BUNNY", 13), 13)
```
    - Add your answer to the variable `t1d = `



_Points:_ 4

In [ ]:
t1a = ...

t1b = ...

t1c = ...

t1d = ...

In [ ]:
grader.check("q1")

### Converting between characters and their numeric representation

As part of the encryption or decryption process we will shift each of the letters.

A few assumptions:
our shift method will only be passed UPPER CASE letters in [A-Z], no spaces, punctuation, or lower case letters, etc.

Note that:

     - `chr(value)` method can convert an integer ordinal value into the cooresponding character (length 1 string). 

     - `ord(char)` method can convert a character into its ordinal value. 

For example: 

```python
    #ordinals
    ord("A") # returns 65 meaning A is encoded as the 65th character in the unicode character set
    ord("B") # returns 66 meaning B is encoded as the 66th character in the unicode character set
    ...
    ord("Z") # returns 90 meaning Z is the encoded as the 90th character in the unicode character set

    #characters
    chr(65) # returns "A"
    ...
    chr(90) # returns "Z"

```


## Shifting letters

What is the result of the following pieces of code?

A: Consider the following code:  `chr(ord("A"))`. Enter what is produced by this code into the next code cell stored in the variable `t2a`:


B: Consider the following code: 

```python
ch = "E"
ch = chr(ord(ch) + 9)
```

Indicate the final value stored in variable `ch` in the next code cell by storing it into variable `t2b`


C: There are 26 letters in the alphabet. Consider that `ord(ch) - ord("A")` gives the index within the alphabet of any letter, e.g., if ch="A" then `ord(ch) - ord("A")` gives 0, for "B" we get 1, ..., for "Z" we get 25. 

If we want the letters to "wrap" around, i.e., we want to have A follow after Z then we can use the mod operator. 

The letter "B" should be 2 letters after "Z" (shifting "Z" by 2 gives "B"). 

What does: `((ord("Z") - ord("A")) + 2) % 26 ` evaluate to? Hint: The answer is 1 because: 27 % 26 == 1 

Now if you add ord("A") to the above value (1) and convert to a character you will get:

`chr( ((ord("Z") - ord("A")) + 2) % 26 + ord("A"))` Enter your answer for this last line into the below code cell next to `t2c =`


_Points:_ 2

In [ ]:
t2a = ...

t2b = ...

t2c = ...

In [ ]:
grader.check("q2")

## Write the shift method

If we think about encrypting a whole message we might realize if we had a method capable of encrypting a single letter we could call this method on each letter in the message that we want to encrypt and we would solve the task.


Do not forget we want our shifts to wrap around the end of the alphabet so "Z" shifted by 1 wraps back to "A"

another example: Y shifted by 2 should give A

Complete the shift method:


_Points:_ 2

In [ ]:
def shift(ch, n):
    """
    shift a single letter string, ch by n ordinal values wrapping 
    values beyond Z back around to A.

    ch: a character (single letter string) within [A-Z]
    n: integer amount to shift ch by

    return: the shifted character (single letter string)
    """

    ...
    
    

In [ ]:
grader.check("q3")

## Encode strings 

Now we can encode a string by shifting each of the letters in the string by a given amount

Complete the below function `encode`, which accepts as arguments a string (the msg to encode) and an integer (the amount to shift each letter). 

The function should call your above shift method to encode each letter and store the encoded letter into the `encoded_str` which is returned from the function.




_Points:_ 2

In [ ]:
def encode(msg, n):
    """ encode a msg by shifting each letter in the string and returning the result 

        msg: a string representing the msg to encode 
             the string is only comprised of CAPITOL letters in [A-Z]
        n: an integer representing the amount to shift the letters by
    """
    #somewhere to build your solution
    encoded_str = ""
    ...

    return encoded_str
        




In [ ]:
grader.check("q4")

## Decode 

To decode an encoded message we can call our encode function with a negative integer. i.e., shift the letters to the left instead of the right. 

Complete the below `decode` function so that it calls and returns the result of our above `encode` function but with negative `n`

In [ ]:
def decode(encoded_msg, n):
    """
        decode the encoded message by shifting each character n spots to the left

        encoded_msg: an encoded string containing only CAPITOL letters
        n: the amount of shift used in the original encoding of the message

        return: the decoded string

    """
    #replace the ellipsis with your solution
    return ...



In [ ]:
# the below call should give you back "BUNNY"
decode(encode("BUNNY",17), 17)

In [ ]:
# the following should be equivalent (True)
encode(encode("SLEEP",9),-9) == decode(encode("SLEEP",9),9)

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

If you visit the link from the course notes you'll come back to the original version of this file each time.

Ensure you run all the cells in order from the top to the bottom
Check to ensure all the tests are passing - run each of the cells containing grader.check ...
After running all the cells (top to bottom) and ensuring the tests are as you want Save your file to your local drive
File->Download->Download.ipynb and then upload the .ipynb file to Moodle under the appropriate Homework submission

Note that your browswer may save your .ipynb file into your Downloads folder or other.

Download and submit the .ipynb version of this file and submit to the Moodle link for Homework submission before the deadline to ensure your grade is recorded.

If you have any issues with downloading or submitting your file just email your professor to ask for help.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
